# Using Unlabeled Data with Self-Supervised Learning

## Part 2 of 2: Finetune the ResNet We Pretrained via Self-Supervised Learning

- Loading a model from the PyTorch Hub: https://pytorch.org/docs/stable/hub.html

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchmetrics
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

torch.manual_seed(123)


In [2]:
model = torch.hub.load('pytorch/vision', 'resnet18', weights=None)
model.fc = nn.Sequential(
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 256),  # must match the architecture used during pretraining
)
model.load_state_dict(torch.load("simclr-resnet18.pt", map_location="cpu"))

# Swap projection head for a linear classifier
model.fc = nn.Linear(512, 10)

Using cache found in /home/zeus/.cache/torch/hub/pytorch_vision_main


In [3]:
# ## Data — standard supervised transforms (no SimCLR augmentation)

# %%
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_full   = torchvision.datasets.CIFAR10(root="data/", train=True,  download=True,  transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root="data/", train=False, download=True,  transform=transform)

indices       = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(123))
train_dataset = Subset(train_full, indices[:45000])
val_dataset   = Subset(train_full, indices[45000:])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [4]:
def train_one_epoch(model, loader, optimizer, criterion, acc_metric, device):
    model.train()
    acc_metric.reset()
    total_loss = 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        acc_metric.update(logits, labels)

    return total_loss / len(loader.dataset), acc_metric.compute().item()


def evaluate(model, loader, criterion, acc_metric, device):
    model.eval()
    acc_metric.reset()
    total_loss = 0.0

    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item() * labels.size(0)
            acc_metric.update(logits, labels)

    return total_loss / len(loader.dataset), acc_metric.compute().item()

In [5]:
torch.manual_seed(123)
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
val_acc   = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

writer = SummaryWriter(log_dir="logs/simclr-finetune")

dummy = torch.zeros(1, 3, 128, 128).to(device)
writer.add_graph(model, dummy)

In [6]:
MAX_EPOCHS = 50

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, train_acc, device)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, val_acc, device)

    writer.add_scalars("Loss",     {"train": tr_loss, "val": vl_loss}, epoch)
    writer.add_scalars("Accuracy", {"train": tr_acc,  "val": vl_acc},  epoch)

    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

writer.close()

Epoch 01/50 | Train Loss: 2.3580  Acc: 0.2839 | Val Loss: 1.6582  Acc: 0.3696
Epoch 02/50 | Train Loss: 1.4358  Acc: 0.4731 | Val Loss: 1.2786  Acc: 0.5358
Epoch 03/50 | Train Loss: 1.1610  Acc: 0.5843 | Val Loss: 1.1357  Acc: 0.6008
Epoch 04/50 | Train Loss: 0.9350  Acc: 0.6698 | Val Loss: 1.0190  Acc: 0.6406
Epoch 05/50 | Train Loss: 0.7571  Acc: 0.7342 | Val Loss: 0.8258  Acc: 0.7120
Epoch 06/50 | Train Loss: 0.6284  Acc: 0.7823 | Val Loss: 0.8552  Acc: 0.7054
Epoch 07/50 | Train Loss: 0.5164  Acc: 0.8181 | Val Loss: 0.7320  Acc: 0.7518
Epoch 08/50 | Train Loss: 0.4256  Acc: 0.8508 | Val Loss: 0.6968  Acc: 0.7644
Epoch 09/50 | Train Loss: 0.3517  Acc: 0.8764 | Val Loss: 0.7962  Acc: 0.7448
Epoch 10/50 | Train Loss: 0.2969  Acc: 0.8941 | Val Loss: 0.8725  Acc: 0.7328
Epoch 11/50 | Train Loss: 0.2334  Acc: 0.9190 | Val Loss: 1.0664  Acc: 0.7196
Epoch 12/50 | Train Loss: 0.2072  Acc: 0.9280 | Val Loss: 0.7697  Acc: 0.7780
Epoch 13/50 | Train Loss: 0.1860  Acc: 0.9338 | Val Loss: 0.9122

In [ ]:
test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
te_loss, te_acc = evaluate(model, test_loader, criterion, test_acc, device)
print(f"\nTest Loss: {te_loss:.4f} | Test Accuracy: {te_acc:.4f}")